# Module 02 — Structured Output

**Goal:** Get reliable, parseable JSON from an LLM — not free-form text.

This is the foundation of the DB Agent's `llm.py` + `models.py`.

By the end of this notebook you will know:
- Why free-form text output breaks production code
- Two ways to get JSON: prompt engineering vs JSON mode
- How Pydantic validates the structure after parsing

---
**Run each cell in order (Shift+Enter).**

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Load .env from the repo root (two levels up) if it exists
load_dotenv(Path("../../.env"))

# Defaults to local Ollama — override by setting these in your .env
BASE_URL = os.getenv("LLM_BASE_URL", "http://localhost:11434/v1")
API_KEY  = os.getenv("LLM_API_KEY",  "ollama")   # Ollama ignores this; required by the client
MODEL    = os.getenv("LLM_MODEL",    "llama3.2:1b")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

print(f"Model:    {MODEL}")
print(f"Endpoint: {BASE_URL}")

Model:    llama3.2:1b
Endpoint: http://localhost:11434/v1


## The problem: free-form output is unparseable at scale

Ask the model to "generate SQL" with no format constraint and you get... whatever it feels like.

In [2]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Write SQL to count rows in a table called 'users'."}
    ],
)

raw = response.choices[0].message.content
print(repr(raw))      # show exactly what came back, including newlines

"To get the number of rows in a specific table, you can use the following SQL query:\n\n```sql\nSELECT COUNT(*) FROM users;\n```\n\nThis will return the total number of rows in the 'users' table. If the table has no values (i.e., it's an empty table), it will return 0."


The model might return:
- Plain SQL: `SELECT count(*) FROM users`
- SQL in a markdown fence: ` ```sql\nSELECT ...\n``` `
- SQL with prose: `"Here's the query: SELECT count(*) FROM users"`
- A different phrasing every time

Writing a parser that handles all these variations is painful and brittle.
The fix: **tell the model what format to use.**

## Approach 1 — Prompt engineering

Put the desired format in the system prompt.
Works everywhere, but the model can still ignore it (especially smaller models).

In [4]:
import json

SYSTEM = """
Always respond with a JSON object in exactly this format:
{"sql": "<your SELECT statement>", "explanation": "<one sentence>"}
Do not include any text outside the JSON object.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": "Count rows in a table called 'users'."}
    ],
    temperature=0,
)

raw = response.choices[0].message.content
print("Raw output:", raw)
print()
print("Parsed:", json.loads(raw))

Raw output: {"sql": "SELECT COUNT(*) FROM users", "explanation": "This SQL statement counts the number of rows in the 'users' table."}

Parsed: {'sql': 'SELECT COUNT(*) FROM users', 'explanation': "This SQL statement counts the number of rows in the 'users' table."}


## Approach 2 — JSON mode (`response_format`)

A stronger API-level guarantee: the model's output will always be valid JSON.
You still need the system prompt to tell the model *which fields* to include.

> **Note:** Not all providers support `response_format`. OpenAI, Groq, and recent Ollama builds do.

In [5]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": "Count rows in a table called 'users'."}
    ],
    temperature=0,
    response_format={"type": "json_object"},  # ← JSON mode
)

raw = response.choices[0].message.content
data = json.loads(raw)   # guaranteed to succeed
print(data)
print(f"sql field:         {data['sql']}")
print(f"explanation field: {data['explanation']}")

{'sql': 'SELECT COUNT(*) FROM users', 'explanation': "This SQL statement counts the number of rows in the 'users' table."}
sql field:         SELECT COUNT(*) FROM users
explanation field: This SQL statement counts the number of rows in the 'users' table.


## Step 3 — Validate with Pydantic

JSON is valid, but `json.loads()` still returns a plain `dict`.
Pydantic turns it into a typed object and validates the fields.

In [7]:
from pydantic import BaseModel, ValidationError

class SQLResponse(BaseModel):
    sql: str
    explanation: str


# Happy path
result = SQLResponse(**data)
print(f"Type: {type(result)}")
print(f"sql:  {result.sql}")
print(f"explanation: {result.explanation}")
print()

# What happens with missing/wrong fields?
try:
    bad = SQLResponse(**{"sql": "SELECT 1"})  # missing 'explanation'
except ValidationError as e:
    print("Validation error caught:")
    print(e)

Type: <class '__main__.SQLResponse'>
sql:  SELECT COUNT(*) FROM users
explanation: This SQL statement counts the number of rows in the 'users' table.

Validation error caught:
1 validation error for SQLResponse
explanation
  Field required [type=missing, input_value={'sql': 'SELECT 1'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


## Step 4 — Defensive parsing (the real-world version)

Even with JSON mode, some models (especially smaller local ones) wrap output in
markdown fences like ` ```json ... ``` `. Here's the defensive parser from
`streamlit_app/llm.py` — it handles these edge cases.

In [9]:
import re

def parse_sql_response(raw: str) -> SQLResponse:
    """Parse LLM output into SQLResponse, handling common formatting issues."""
    # Strip markdown fences
    text = re.sub(r"```(?:json)?\s*", "", raw).strip().rstrip("`").strip()

    # Fast path: whole response is valid JSON
    try:
        data = json.loads(text)
        return SQLResponse(sql=data.get("sql", "").strip(),
                           explanation=data.get("explanation", "").strip())
    except json.JSONDecodeError:
        pass

    # Fallback: find first {...} block in surrounding prose
    match = re.search(r"\{[^{}]*\}", text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in LLM response:\n{raw}")

    data = json.loads(match.group())
    return SQLResponse(sql=data.get("sql", "").strip(),
                       explanation=data.get("explanation", "").strip())


# Test with various formats the model might return:
test_cases = [
    '{"sql": "SELECT 1", "explanation": "Clean JSON"}',
    '```json\n{"sql": "SELECT 2", "explanation": "Fenced JSON"}\n```',
    'Here you go: {"sql": "SELECT 3", "explanation": "JSON in prose"}',
]

for case in test_cases:
    result = parse_sql_response(case)
    print(f"sql={result.sql!r:15}  explanation={result.explanation!r}")

sql='SELECT 1'       explanation='Clean JSON'
sql='SELECT 2'       explanation='Fenced JSON'
sql='SELECT 3'       explanation='JSON in prose'


## Step 5 — Putting it all together

Here's the complete flow that `streamlit_app/llm.py` implements:
LLM call → raw text → defensive parse → Pydantic model.

In [10]:
def generate_sql(question: str, schema_text: str) -> SQLResponse:
    """Ask the LLM to generate SQL and return a validated SQLResponse."""
    system = """
You are a SQL assistant. Return only valid JSON with this exact structure:
{"sql": "<SELECT statement or empty string>", "explanation": "<one sentence>"}
Only generate SELECT statements. Never use DROP, DELETE, UPDATE, INSERT, etc.
"""
    user = f"Schema:\n{schema_text}\n\nQuestion: {question}"

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        temperature=0,
    )
    return parse_sql_response(response.choices[0].message.content)


schema = "users: id (INTEGER), name (TEXT), email (TEXT), created_at (DATETIME)"
result = generate_sql("How many users signed up?", schema)

print(f"SQL:         {result.sql}")
print(f"Explanation: {result.explanation}")

SQL:         SELECT COUNT(*) FROM users
Explanation: Counts the number of rows in the users table


## Summary

| Technique | Pros | Cons |
|-----------|------|------|
| Free-form output | No setup | Unparseable, inconsistent |
| Prompt engineering | Works everywhere | Model can ignore it |
| JSON mode | API-level guarantee | Not all providers support it |
| Pydantic validation | Type-safe, catches bad models | Requires defining schemas |

**In production, use all three together** (prompt + JSON mode + Pydantic).

Look at how this maps to the DB Agent:
- `prompts.py` → sets the JSON format in the system prompt
- `llm.py` → makes the API call + defensive parsing
- `models.py` → defines `SQLResponse` (the Pydantic model)

**Next:** Module 03 — Tool Use. Instead of just generating text (or JSON), the LLM will *call functions* and decide what data to fetch.